# Powerlifting Gold Layer

Builds the business-ready analytical layer in `workspace.powerlifting_gold`: `dim_meets`, `fact_meet_results`, and pre-computed `agg_meet_summary` / `agg_lifter_summary` tables for reporting and BI.

In [0]:
%sql
-- Gold layer: business-ready analytical tables for reporting and BI.
-- Joins silver.openpowerlifting with silver.meets, adds derived dimensions,
-- and pre-computes common aggregations.

CREATE SCHEMA IF NOT EXISTS workspace.powerlifting_gold;

-- dim_meets: enriched meet dimension with temporal and geographic attributes.
-- ParticipantCount is precomputed for fast meet-level filtering.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.dim_meets AS
SELECT
  m.MeetID,
  m.MeetPath,
  m.Federation,
  m.MeetDate,
  YEAR(m.MeetDate)         AS MeetYear,
  FLOOR(YEAR(m.MeetDate) / 10) * 10 AS MeetDecade,
  m.MeetCountry,
  m.MeetState,
  m.MeetTown,
  m.MeetName,
  COUNT(op.MeetID)         AS ParticipantCount
FROM workspace.powerlifting_silver.meets m
LEFT JOIN workspace.powerlifting_silver.openpowerlifting op ON m.MeetID = op.MeetID
GROUP BY m.MeetID, m.MeetPath, m.Federation, m.MeetDate,
         m.MeetCountry, m.MeetState, m.MeetTown, m.MeetName;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- fact_meet_results: enriched one-row-per-lifter-per-meet fact table.
-- Denormalises meet info for convenience, adds EventCategory, AgeCategory,
-- and StrengthRatio (total / bodyweight) for direct analytical use.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.fact_meet_results AS
SELECT
  op.MeetID,
  op.Name,
  op.Sex,
  op.SexLabel,
  op.Equipment,
  op.Age,
  op.Division,
  op.BodyweightKg,
  op.WeightClassKg,
  op.BestSquatKg,
  op.BestBenchKg,
  op.BestDeadliftKg,
  op.Squat4Kg,
  op.Bench4Kg,
  op.Deadlift4Kg,
  op.TotalKg,
  op.Place,
  op.FinishStatus,
  op.Wilks,
  -- Denormalised meet columns
  m.MeetDate,
  m.MeetName,
  m.Federation,
  m.MeetCountry,
  m.MeetState,
  m.MeetTown,
  YEAR(m.MeetDate)         AS MeetYear,
  FLOOR(YEAR(m.MeetDate) / 10) * 10 AS MeetDecade,
  -- Event category derived from which lifts were attempted
  CASE
    WHEN op.BestSquatKg IS NOT NULL AND op.BestBenchKg IS NOT NULL AND op.BestDeadliftKg IS NOT NULL THEN 'Full Power'
    WHEN op.BestBenchKg IS NOT NULL AND op.BestSquatKg IS NULL AND op.BestDeadliftKg IS NULL THEN 'Bench Only'
    WHEN op.BestSquatKg IS NOT NULL AND op.BestBenchKg IS NULL AND op.BestDeadliftKg IS NOT NULL THEN 'Squat-Deadlift'
    WHEN op.BestSquatKg IS NOT NULL AND op.BestBenchKg IS NULL AND op.BestDeadliftKg IS NULL THEN 'Squat Only'
    WHEN op.BestDeadliftKg IS NOT NULL AND op.BestSquatKg IS NULL AND op.BestBenchKg IS NULL THEN 'Deadlift Only'
    ELSE 'Other'
  END AS EventCategory,
  -- Strength ratio: total lifted relative to bodyweight
  CASE WHEN op.BodyweightKg > 0 AND op.TotalKg IS NOT NULL
       THEN ROUND(op.TotalKg / op.BodyweightKg, 3)
  END AS StrengthRatio,
  -- Age category (IPF-style bands)
  CASE
    WHEN op.Age < 18  THEN 'Youth'
    WHEN op.Age BETWEEN 18 AND 23  THEN 'Junior'
    WHEN op.Age BETWEEN 24 AND 34  THEN 'Open'
    WHEN op.Age BETWEEN 35 AND 49  THEN 'Sub-Master'
    WHEN op.Age >= 50 THEN 'Master'
    ELSE 'Unknown'
  END AS AgeCategory,
  CASE WHEN op.FinishStatus = 'Finished' THEN TRUE ELSE FALSE END AS IsCompleted
FROM workspace.powerlifting_silver.openpowerlifting op
LEFT JOIN workspace.powerlifting_silver.meets m ON op.MeetID = m.MeetID;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- agg_meet_summary: one row per meet with participant counts, lift stats,
-- sex breakdown, equipment distribution, and the top lifter by TotalKg.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.agg_meet_summary AS
SELECT
  m.MeetID,
  m.MeetName,
  m.MeetDate,
  YEAR(m.MeetDate) AS MeetYear,
  m.Federation,
  m.MeetCountry,
  COUNT(*)                                       AS TotalParticipants,
  SUM(CASE WHEN op.Sex = 'M' THEN 1 ELSE 0 END)  AS MaleParticipants,
  SUM(CASE WHEN op.Sex = 'F' THEN 1 ELSE 0 END) AS FemaleParticipants,
  ROUND(AVG(op.TotalKg), 2)                       AS AvgTotalKg,
  MAX(op.TotalKg)                                 AS MaxTotalKg,
  ROUND(AVG(op.Wilks), 2)                          AS AvgWilks,
  MAX(op.Wilks)                                   AS MaxWilks,
  SUM(CASE WHEN op.FinishStatus = 'Finished' THEN 1 ELSE 0 END)       AS FinishedCount,
  SUM(CASE WHEN op.FinishStatus = 'Disqualified' THEN 1 ELSE 0 END)  AS DisqualifiedCount,
  MAX_BY(op.Name, op.TotalKg)                     AS TopLifterName,
  SUM(CASE WHEN op.Equipment = 'Raw'       THEN 1 ELSE 0 END) AS RawCount,
  SUM(CASE WHEN op.Equipment = 'Single-ply' THEN 1 ELSE 0 END) AS SinglePlyCount,
  SUM(CASE WHEN op.Equipment = 'Multi-ply' THEN 1 ELSE 0 END) AS MultiPlyCount,
  SUM(CASE WHEN op.Equipment = 'Wraps'      THEN 1 ELSE 0 END) AS WrapsCount
FROM workspace.powerlifting_silver.openpowerlifting op
JOIN workspace.powerlifting_silver.meets m ON op.MeetID = m.MeetID
GROUP BY m.MeetID, m.MeetName, m.MeetDate, YEAR(m.MeetDate), m.Federation, m.MeetCountry;

-- agg_lifter_summary: one row per lifter with career stats, personal records,
-- federation diversity, and favourite equipment type.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.agg_lifter_summary AS
WITH lifter_agg AS (
  SELECT
    op.Name,
    op.Sex,
    op.SexLabel,
    COUNT(DISTINCT op.MeetID)        AS TotalMeets,
    MIN(m.MeetDate)                  AS FirstMeetDate,
    MAX(m.MeetDate)                  AS LastMeetDate,
    DATEDIFF(MAX(m.MeetDate), MIN(m.MeetDate)) AS CareerSpanDays,
    MAX(op.TotalKg)                  AS BestTotalKg,
    MAX(op.BestSquatKg)              AS BestSquatKg,
    MAX(op.BestBenchKg)              AS BestBenchKg,
    MAX(op.BestDeadliftKg)           AS BestDeadliftKg,
    ROUND(AVG(op.Wilks), 2)          AS AvgWilks,
    MAX(op.Wilks)                    AS BestWilks,
    COUNT(DISTINCT m.Federation)     AS DistinctFederations
  FROM workspace.powerlifting_silver.openpowerlifting op
  JOIN workspace.powerlifting_silver.meets m ON op.MeetID = m.MeetID
  GROUP BY op.Name, op.Sex, op.SexLabel
),
equipment_ranked AS (
  SELECT
    Name, Sex, Equipment,
    ROW_NUMBER() OVER (PARTITION BY Name, Sex ORDER BY COUNT(*) DESC) AS eq_rn
  FROM workspace.powerlifting_silver.openpowerlifting
  GROUP BY Name, Sex, Equipment
)
SELECT
  la.*,
  er.Equipment AS FavouriteEquipment
FROM lifter_agg la
LEFT JOIN equipment_ranked er ON la.Name = er.Name AND la.Sex = er.Sex AND er.eq_rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify gold layer: row counts and spot-checks
SELECT 'dim_meets' AS table_name, COUNT(*) AS row_count FROM workspace.powerlifting_gold.dim_meets
UNION ALL
SELECT 'fact_meet_results', COUNT(*) FROM workspace.powerlifting_gold.fact_meet_results
UNION ALL
SELECT 'agg_meet_summary', COUNT(*) FROM workspace.powerlifting_gold.agg_meet_summary
UNION ALL
SELECT 'agg_lifter_summary', COUNT(*) FROM workspace.powerlifting_gold.agg_lifter_summary;

-- EventCategory distribution
SELECT EventCategory, COUNT(*) AS cnt
FROM workspace.powerlifting_gold.fact_meet_results
GROUP BY 1
ORDER BY 2 DESC;

-- AgeCategory distribution
SELECT AgeCategory, COUNT(*) AS cnt
FROM workspace.powerlifting_gold.fact_meet_results
GROUP BY 1
ORDER BY 2 DESC;

-- Confirm fact table join coverage (no unmatched meets)
SELECT
  SUM(CASE WHEN MeetDate IS NULL THEN 1 ELSE 0 END) AS unmatched_meets,
  SUM(CASE WHEN MeetDate IS NOT NULL THEN 1 ELSE 0 END) AS matched_meets
FROM workspace.powerlifting_gold.fact_meet_results;

-- Top 10 lifters by best TotalKg
SELECT Name, SexLabel, BestTotalKg, BestSquatKg, BestBenchKg, BestDeadliftKg,
       BestWilks, TotalMeets, FavouriteEquipment
FROM workspace.powerlifting_gold.agg_lifter_summary
ORDER BY BestTotalKg DESC NULLS LAST
LIMIT 10;

Name,SexLabel,BestTotalKg,BestSquatKg,BestBenchKg,BestDeadliftKg,BestWilks,TotalMeets,FavouriteEquipment
Dave Hoff,Male,1365.31,560.19,455.86,374.21,779.38,23,Multi-ply
Donnie Thompson,Male,1360.78,573.79,430.91,356.07,735.88,4,Multi-ply
AJ Roberts,Male,1295.01,546.58,412.77,369.68,725.48,12,Multi-ply
Blaine Sumner,Male,1272.5,505.0,410.0,370.0,692.21,20,Single-ply
Shane Hammock,Male,1270.06,508.02,396.89,399.16,714.43,11,Multi-ply
Jake Anderson,Male,1270.06,521.63,376.48,396.89,707.79,14,Multi-ply
Jeff Frank,Male,1257.5,552.5,367.5,340.0,675.9,15,Multi-ply
Josh Conley,Male,1249.65,489.88,374.21,408.23,686.71,3,Multi-ply
Brian Carroll,Male,1238.31,537.51,374.21,353.8,707.94,13,Multi-ply
Chris Janek,Male,1236.04,501.22,356.07,381.02,690.92,9,Multi-ply
